# Huntington disease: DREAMwalk candidate prediction

**Pipeline: all known DDA positives + ten link-level negative sets → ten XGBoost classifiers → predict unlabeled HD–drug pairs → rank mean scores and report SD → literature review.**

Use ten negative-set predictions following the paper's case-study design. **Do not hold out all HD labels in this workflow.** `NS_SEEDS = 0–9` controls downstream **link-level NS** only. This is not ten-fold CV or ten embedding retraining runs.

Reuse the completed **MeSH2024, embedding seed 0, τ=0.3, modified pp=1** embedding: first-hit teleport, positive-output-type Skip-gram NS and current-positive collision filtering. The prediction design follows the case-study workflow; the embedding remains the project's combined variant.

Select the existing **DREAMwalk Python 3.8 kernel** and run from top to bottom or use **Run All**. Section 5 starts classifier training. Keep historical holdout outputs separate. This publication copy has cleared outputs; the README links to the completed ten-set result snapshot.


## 1. Paths and existing environment

Interpreter: `00.Reproduction/00.DREAMwalk_reproduction/environment/.conda-env/bin/python`.

In VS Code select **DREAMwalk MSI (Python 3.8)**. Reuse that environment without a new installation. Set `REPO_OVERRIDE` if the notebook starts outside the repository. Required local inputs are documented in the root README.


In [ ]:
from pathlib import Path
import sys

REPO_OVERRIDE = None
RUN_NAME = 'hd_mesh2024_fixed_ns_seed0_9'
NS_SEEDS = list(range(10))

start = Path(REPO_OVERRIDE).expanduser().resolve() if REPO_OVERRIDE else Path.cwd().resolve()
REPO = next((p for p in [start, *start.parents]
             if (p/'00.Reproduction/00.DREAMwalk_reproduction').is_dir()), None)
if REPO is None:
    raise FileNotFoundError('Repository not found; set REPO_OVERRIDE.')
CASE = REPO/'01.case_study/01.HD'
DREAMWALK = REPO/'00.Reproduction/00.DREAMwalk_reproduction'
EXPECTED_PYTHON = DREAMWALK/'environment/.conda-env/bin/python'
if sys.version_info[:2] != (3, 8):
    raise RuntimeError(f'Select the DREAMwalk Python 3.8 kernel: {EXPECTED_PYTHON}')
if not RUN_NAME or any(c not in 'abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789_-' for c in RUN_NAME):
    raise ValueError('RUN_NAME accepts only letters, digits, underscores and hyphens.')
RESULTS = CASE/'results'/RUN_NAME
NOTEBOOK = CASE/'notebooks/HD_DREAMwalk_drug_prediction.ipynb'
sys.dont_write_bytecode = True
sys.path.insert(0, str(CASE/'src'))

import hashlib
import json
import platform
import random
import time
import numpy as np
import pandas as pd
from IPython.display import display
from xgboost import XGBClassifier
from hd_prediction import (
    DISEASE, load_inputs, pair_features, sha256, read_rows,
    save_rows, open_run, check_stage, finish_stage,
)
from hd_paper_prediction import NEGATIVE_SEEDS, sample_negative_sets, aggregate_scores
assert NS_SEEDS == list(NEGATIVE_SEEDS)

print('Interpreter:', sys.executable)
print('Link-level NS seeds:', NS_SEEDS)
print('Case-study outputs:', RESULTS)


## 2. Read existing embeddings and all known positives

Check the embedding completion record, SHA-256, shared inputs and environment. Reuse the completed experiment's XGBoost parameters and `drug embedding − disease embedding` features.

For historical compatibility, the input helper returns non-HD and HD rows separately. **Immediately combine both groups again.** Each classifier uses all 5,926 positives, including five known HD associations, plus 5,926 sampled negatives.


In [ ]:
vectors, node_types, drug_names, other_rows, hd_reference_rows, provenance = load_inputs(DREAMWALK)
reference_rows = sorted(other_rows + hd_reference_rows, key=lambda row: row['pair_index'])
positive_pairs = [(row['drug'], row['disease']) for row in reference_rows if row['label'] == 1]
seed0_negatives = [(row['drug'], row['disease']) for row in reference_rows if row['label'] == 0]
known_hd = {drug for drug, disease in positive_pairs if disease == DISEASE}
drugs = sorted(node for node, kind in node_types.items() if kind == 'drug')
XGB_PARAMS = dict(provenance['xgboost'])
assert XGB_PARAMS['seed'] == 0

print('Embedding run:', provenance['embedding_run'])
print('Embedding SHA256:', provenance['inputs']['embedding']['sha256'])
print(f'{len(positive_pairs):,} training positives per classifier; {len(known_hd)} include HD.')
print(f'{len(drugs):,} drugs; {len(drugs) - len(known_hd):,} unlabeled HD candidates.')
display(pd.DataFrame([{
    'mesh_year': provenance['semantic']['mesh_year'],
    'embedding_seed': provenance['settings']['seed'],
    'tau': provenance['settings']['tp_factor'], 'pp': provenance['settings']['pp'],
    'teleport_selector': provenance['settings']['teleport_selector'],
    'Skip-gram_NS_pool': provenance['settings']['ns_pool'],
    'positive_collision_filter': provenance['settings']['reject_positive_output'],
}]))
display(pd.DataFrame([XGB_PARAMS]).T.rename(columns={0: 'value'}))


## 3. Fixed prediction protocol

- Link-level NS seeds **0–9**. Preserve the saved seed-0 pairs. Sample seeds 1–9 sequentially from remaining unknown drug–disease pairs. Each set has a 1:1 positive/negative ratio.
- The ten negative sets are **pairwise disjoint**. This applies the paper Methods' non-overlap convention locally; the case-study paragraph itself specifies different negative sets.
- Every classifier uses all known positives, **without HD-label holdout**. Fix XGBoost, Python and NumPy training seeds at 0; change only the negative set.
- Keep one combined-variant embedding fixed. SD measures link-negative-set variation and excludes embedding-retraining variation.
- The complete author case-study refit script is unavailable. Full-known-positive classifier refitting is an explicit local implementation choice.

Record input, code and parameter fingerprints. Matching runs resume completed stages. Source, configuration or notebook-code changes require a new `RUN_NAME` when old outputs exist. English message translations change notebook-code hashes; do not overwrite historical fingerprints to bypass this check.


In [ ]:
notebook_document = json.loads(NOTEBOOK.read_text(encoding='utf-8'))
code_cells = [''.join(cell['source']) for cell in notebook_document['cells'] if cell['cell_type'] == 'code']
identity = {
    'disease': DISEASE, 'disease_name': 'Huntington disease',
    'protocol': 'ten negative-set predictions; full-known-positive classifier refit; no HD label holdout',
    'reference': 'Bang et al. 2023 case study and Methods; doi:10.1038/s41467-023-39301-y',
    'ns_seeds': NS_SEEDS,
    'negative_sampling': 'sorted drugs x sorted diseases minus ALL known positives; balanced sets; '
                         'seed0 saved sample; seeds1-9 Random(seed).sample(remaining); no cross-set overlap',
    'label_order': 'sorted positive pairs followed by each saved negative list',
    'training_positives_per_model': len(positive_pairs),
    'training_negatives_per_model': len(positive_pairs),
    'classifier_seed': 0,
    'candidate_scope': 'all embedded MSI drugs; exclude known HD positives from candidate ranking',
    'aggregation': 'mean score across all 10 sets; sample SD ddof=1; ties by drug ID',
    'uncertainty_scope': 'negative-set variability conditional on one fixed corrected embedding',
    'source': provenance,
    'helper_sha256': {name: sha256(CASE/'src'/name)
                      for name in ['hd_prediction.py', 'hd_paper_prediction.py']},
    'notebook_code_sha256': hashlib.sha256(json.dumps(code_cells, ensure_ascii=False).encode()).hexdigest(),
    'python': sys.version, 'platform': platform.platform(),
}
fingerprint = open_run(RESULTS, identity)
print('Run fingerprint:', fingerprint)
print('Configuration:', RESULTS/'run_config.json')


## 4. Save link-level negatives for seeds 0–9

The candidate pool is all MSI drug–disease combinations excluding every known positive pair. A sampled unknown pair is not evidence of biological inactivity.

Save actual negative IDs for each seed. If an HD candidate is sampled as a training negative, retain its prediction and mark the exposure. Seed 0 must match the saved pairs and their order exactly.


In [ ]:
if check_stage(RESULTS, 'sampling', fingerprint):
    negative_sets = {
        seed: [(row['drug'], row['disease'])
               for row in read_rows(RESULTS/f'seed{seed:02d}'/'negative_pairs.csv')]
        for seed in NS_SEEDS
    }
    print('Reusing all ten saved negative sets.')
else:
    negative_sets = sample_negative_sets(positive_pairs, seed0_negatives, node_types)
    files = ['positive_pairs.csv', 'negative_sampling_summary.csv']
    save_rows(RESULTS/'positive_pairs.csv',
              [dict(drug=drug, disease=disease, label=1) for drug, disease in positive_pairs])
    for seed in NS_SEEDS:
        directory = RESULTS/f'seed{seed:02d}'
        directory.mkdir(exist_ok=True)
        save_rows(directory/'negative_pairs.csv',
                  [dict(drug=drug, disease=disease, label=0) for drug, disease in negative_sets[seed]])
        files.append(f'seed{seed:02d}/negative_pairs.csv')
    summary = [dict(ns_seed=seed, positives=len(positive_pairs), negatives=len(negative_sets[seed]),
                    hd_training_negatives=sum(disease == DISEASE for _, disease in negative_sets[seed]))
               for seed in NS_SEEDS]
    save_rows(RESULTS/'negative_sampling_summary.csv', summary)
    finish_stage(RESULTS, 'sampling', fingerprint, files)
assert negative_sets[0] == seed0_negatives
assert all(len(negative_sets[seed]) == len(positive_pairs) for seed in NS_SEEDS)
assert len(set(pair for pairs in negative_sets.values() for pair in pairs)) == len(NS_SEEDS)*len(positive_pairs)
assert not set(positive_pairs).intersection(pair for pairs in negative_sets.values() for pair in pairs)
display(pd.read_csv(RESULTS/'negative_sampling_summary.csv'))


## 5. Train or reuse ten XGBoost classifiers

**Running this section starts classifier training.** Each model uses the same complete positive set and its own negative set. No CV, parameter selection or new embedding generation is performed.

Immediately save `classifier.json` and its completion record after each seed. On restart, rerun setup and reuse completed classifiers.


In [ ]:
if not check_stage(RESULTS, 'sampling', fingerprint):
    raise RuntimeError('Complete Section 4 first.')
for seed in NS_SEEDS:
    directory = RESULTS/f'seed{seed:02d}'
    model_path = directory/'classifier.json'
    if check_stage(directory, 'training', fingerprint):
        print(f'NS seed {seed}: reusing completed classifier.', flush=True)
        continue
    if model_path.exists():
        raise RuntimeError(f'{model_path} already exists without a completion record; refusing to overwrite it.')
    pairs = positive_pairs + negative_sets[seed]
    X_train = pair_features(vectors, pairs)
    y_train = np.asarray([1]*len(positive_pairs) + [0]*len(negative_sets[seed]), dtype=np.int8)
    assert X_train.shape == (len(y_train), provenance['settings']['dimension'])
    random.seed(0)
    np.random.seed(0)
    model = XGBClassifier(**XGB_PARAMS)
    started = time.monotonic()
    print(f'NS seed {seed}: fitting on {len(y_train):,} labelled pairs...', flush=True)
    model.fit(X_train, y_train)
    assert list(model.classes_) == [0, 1]
    temporary_model = directory/'classifier.pending.json'
    model.save_model(str(temporary_model))
    temporary_model.replace(model_path)
    finish_stage(directory, 'training', fingerprint, ['classifier.json', 'negative_pairs.csv'])
    print(f'NS seed {seed}: saved; {time.monotonic() - started:.1f} seconds.', flush=True)


## 6. Score every drug with each classifier

Fix HD and construct `drug embedding − HD embedding` for each of 1,661 drugs. Save each seed's XGBoost positive-class scores immediately.

Also score known HD positives for inspection, but exclude them from the novel-candidate ranking. They participated in classifier training, so their scores are **in-sample**, not independent validation.


In [ ]:
if not check_stage(RESULTS, 'sampling', fingerprint):
    raise RuntimeError('Complete Section 4 first.')
X_hd = pair_features(vectors, [(drug, DISEASE) for drug in drugs])
for seed in NS_SEEDS:
    directory = RESULTS/f'seed{seed:02d}'
    if not check_stage(directory, 'training', fingerprint):
        raise RuntimeError(f'NS seed {seed} classifier is not complete.')
    if check_stage(directory, 'prediction', fingerprint):
        print(f'NS seed {seed}: reusing saved scores.')
        continue
    model = XGBClassifier(**XGB_PARAMS)
    model.load_model(str(directory/'classifier.json'))
    assert list(model.classes_) == [0, 1]
    scores = model.predict_proba(X_hd)[:, 1]
    assert len(scores) == len(drugs) and np.isfinite(scores).all() and ((scores >= 0) & (scores <= 1)).all()
    pool = set(negative_sets[seed])
    rows = [dict(ns_seed=seed, drug=drug, drug_name=drug_names.get(drug) or drug,
                 disease=DISEASE, association_score=float(score), known_in_msi=drug in known_hd,
                 used_as_training_negative=(drug, DISEASE) in pool)
            for drug, score in zip(drugs, scores)]
    save_rows(directory/'drug_scores.csv', rows)
    finish_stage(directory, 'prediction', fingerprint, ['drug_scores.csv', 'classifier.json'])
    print(f'NS seed {seed}: saved {len(rows):,} scores.')


## 7. Aggregate ten predictions: mean, SD and ranking

**Require all seeds 0–9 before aggregation.** Compute each drug's mean score and sample SD (`ddof=1`), then rank by descending mean. The paper does not specify the SD denominator; this implementation explicitly uses sample SD.

`training_negative_count` records how many negative sets contain the HD pair; `training_negative_fraction` gives the corresponding proportion. Pairwise disjoint sampling limits each pair to at most one set. These fields do not remove or otherwise filter candidates.


In [ ]:
per_seed_rows = []
scores_by_seed = {}
for seed in NS_SEEDS:
    directory = RESULTS/f'seed{seed:02d}'
    if not check_stage(directory, 'training', fingerprint) or not check_stage(directory, 'prediction', fingerprint):
        raise RuntimeError(f'NS seed {seed} is incomplete; aggregation requires all ten runs.')
    rows = read_rows(directory/'drug_scores.csv')
    assert [row['drug'] for row in rows] == drugs
    assert all(int(row['ns_seed']) == seed and row['disease'] == DISEASE for row in rows)
    scores_by_seed[seed] = [float(row['association_score']) for row in rows]
    per_seed_rows.extend(rows)

if not check_stage(RESULTS, 'aggregation', fingerprint):
    all_ranked, candidates, known = aggregate_scores(
        drugs, scores_by_seed, drug_names, positive_pairs, negative_sets)
    assert len(candidates) + len(known) == len(drugs) and len(known) == len(known_hd)
    tables = {
        'per_seed_scores.csv': per_seed_rows,
        'all_drugs_ranked.csv': all_ranked,
        'novel_candidates_ranked.csv': candidates,
        'top10_candidates.csv': candidates[:10],
        'top20_candidates.csv': candidates[:20],
        'known_hd_associations_ranked.csv': known,
    }
    for name, rows in tables.items():
        save_rows(RESULTS/name, rows)
    finish_stage(RESULTS, 'aggregation', fingerprint, list(tables))
    print('All ten negative-set predictions aggregated:', RESULTS)
else:
    print('Reusing completed mean/SD rankings:', RESULTS)


## 8. Inspect the top ten candidates and review literature

The paper's case study reviews the top ten predictions. Save the top 20 and full ranking as well, so the scope of subsequent analysis can be selected explicitly.

`mean_score` is a ranking score, **not a calibrated clinical probability**. Novel means absent from the current MSI HD positive labels. A high mean or small SD does not replace efficacy evidence.


In [ ]:
if not check_stage(RESULTS, 'aggregation', fingerprint):
    raise RuntimeError('Complete Section 7 first.')
candidate_table = pd.read_csv(RESULTS/'novel_candidates_ranked.csv')
columns = ['candidate_rank', 'drug', 'drug_name', 'mean_score', 'sd_score',
           'n_negative_sets', 'training_negative_count']
with pd.option_context('display.max_rows', 20, 'display.float_format', '{:.6f}'.format):
    display(candidate_table[columns].head(10))
print('Top 10:', RESULTS/'top10_candidates.csv')
print('Top 20:', RESULTS/'top20_candidates.csv')
print('Full ranking:', RESULTS/'novel_candidates_ranked.csv')


## 9. Inspect known HD associations separately

These five positive pairs trained all ten classifiers. Display them for context without treating their scores as predictive validation. No AUROC is calculated in this section.


In [ ]:
if not check_stage(RESULTS, 'aggregation', fingerprint):
    raise RuntimeError('Complete Section 7 first.')
known_table = pd.read_csv(RESULTS/'known_hd_associations_ranked.csv')
with pd.option_context('display.float_format', '{:.6f}'.format):
    display(known_table[['rank_all', 'drug', 'drug_name', 'mean_score', 'sd_score']])
print('Run manifest:', RESULTS/'run_config.json')


## Method source, historical workflow and limitations

[Bang et al. (2023)](https://www.nature.com/articles/s41467-023-39301-y) predict unlabeled pairs ten times using different negative sets, rank average scores, report SD and examine literature. This notebook applies that prediction design to HD with the project's completed combined-variant embedding.

- Current run: `results/hd_mesh2024_fixed_ns_seed0_9/`. A completed result snapshot is published under the root `reference_results/` directory.
- Historical single-run HD-label holdout: `results/hd_mesh2024_fixed_seed0/` in the original working project. Its outputs and duplicate notebook archive are not included in this code release. Historical rankings are not results of the current protocol.
- Graph incompleteness can introduce false negatives; see the root README's limitations section. Type-aware Skip-gram NS and the link-level NS used here belong to different training stages.
- The complete original case-study fit script and historical benchmark snapshot are unavailable. Record full-positive refitting, negative-set construction and fixed-embedding choices in the run manifest.

Next, select a small set of candidates from the current ranking and review HD literature, mechanisms, clinical/preclinical evidence and negative trials.
